In [6]:
from dotenv import load_dotenv
from anthropic import Anthropic
import os

from rag_helper import RAGHelper
from ingest import load_faq_data, build_index
import json

In [7]:
load_dotenv()  # Load environment variables from .env file

True

In [8]:
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
documents = load_faq_data()
index = build_index(documents)

In [10]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use ONLY the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [11]:
assistant = RAGHelper(
    index=index, llm_client=anthropic_client, instructions=instructions
)

NameError: name 'index' is not defined

In [ ]:
answer = assistant.rag("I just dicovered the course. Can I join it?")
print(answer)

Yes, you can join the course! However, there's an important condition:

If you want to receive a certificate, you'll need to submit your project while the course is still accepting submissions. The course is structured around live cohorts, and project submission and peer review must happen during the active submission window.

You can start learning and working through the materials at your own pace using the available videos and GitHub resources, but to get certified, you need to complete the capstone project and required peer reviews while a live cohort is accepting them.


In [ ]:
# Testing without the database context, just using the LLM directly.
messages = [{"role": "user", "content": "I just dicovered the course. Can I join it?"}]

response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=messages,
)
print(response.content[0].text)

I'd be happy to help! However, I don't have information about which specific course you're referring to or its enrollment details.

To give you accurate information, could you please tell me:

1. **What course are you interested in?** (name, subject, or platform)
2. **Where is it offered?** (university, online platform like Coursera, Udemy, etc.)
3. **When does it start?**

Once you provide


In [ ]:
def search(query):
    """
    Search the FAQ database for relevant context based on the query.
    """
    boost_dict = {
        "question": 3.0,
        "answer": 0.5,
    }  # Boost the question field more than the answer field
    filter_dict = {
        "course": "llm-zoomcamp"
    }  # Filter to only include documents related to the llm-zoomcamp course

    return index.search(
        query, num_results=5, boost_dict=boost_dict, filter_dict=filter_dict
    )

In [ ]:
# JSON Function definition/schema for the search tool
search_tool = {
    "type": "custom",
    "name": "search",
    "description": "Search the FAQ database for relevant context based on the query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to search for in the FAQ database.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [ ]:
messages = [{"role": "user", "content": "I just discovered the course. Can I join it?"}]

response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=messages,
    tools=[search_tool],  # Provide the search tool to the model
    tool_choice={"name": "search", "type": "tool"},
)

In [ ]:
call = response.content[0]

In [ ]:
call

ToolUseBlock(id='toolu_01EenMWimh39rFR3JiTH4dn9', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment requirements'}, name='search', type='tool_use')

In [ ]:
args = call.input
query = args.get("query")
results = search(query)
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '5cc511f85b', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Does the course certificate show the number of course hours?', 'answer': 'No. The certificate does not state a total number of hours.'}, {'id': 'bd31146b0e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'When will the course be offered next?', 'answer': 'Summer 2027.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp

In [ ]:
results = search(**args)

In [ ]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/ll

In [ ]:
results_json = json.dumps(results, indent=2)

In [ ]:
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2027."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "S

In [ ]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.id,
    "output": results_json,
}

In [ ]:
messages.append(call)

In [ ]:
messages.append(function_call_output)

In [ ]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ToolUseBlock(id='toolu_01EenMWimh39rFR3JiTH4dn9', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment requirements'}, name='search', type='tool_use'),
 {'type': 'function_call_output',
  'call_id': 'toolu_01EenMWimh39rFR3JiTH4dn9',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "5cc511f85b",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Does the course certificate show the number of course hours?",\n    "answer": "No. The certificate does not state a total number of hours."\n  },\n  {\n    "id": "bd31146b0e",\n    "course"

In [ ]:
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    tools=[search_tool],
    messages=[
        {
            "role": "user",
            "content": results_json,
        }
    ],
)

In [ ]:
print(response.content[0].text)

I can see you've provided a FAQ database for the LLM Zoomcamp course. I have access to a search function that can query this database to help answer questions about the course.

How can I help you? You can ask me questions about:
- Joining the course
- Certificate requirements
- Course timeline and workflow
- When the course is offered next
- Self-paced options
- Or any other course-related questions

Just let me know what you'd like to know!


In [ ]:
response.usage.input_tokens, response.usage.output_tokens

(1289, 109)

Looping through our tools until the LLM is satisfied. (Agentic Loop)

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
question = "I just discovered the course. Can I join it?"
messages = [
    {"role": "assistant", "content": instructions},
    {"role": "user", "content": question},
]

In [ ]:
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=messages,
    tools=[search_tool],
)

In [ ]:
print(json.dumps(messages, indent=2, default=str))

[
  {
    "role": "assistant",
    "content": "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."
  },
  {
    "role": "user",
    "content": "I just discovered the course. Can I join it?"
  },
  {
    "role": "assistant",
    "content": [
      "TextBlock(citations=None, text=\"I'll help you find information about joining the course! Let me search for details about enrollment and joining policies.\", type='text')",
      "ToolUseBlock(id='toolu_01S54T5w3VwHgGJXccD4sp6Q', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment'}, name='search', type='tool_use

In [ ]:
print(response)

Message(id='msg_011Cede9gw9iX4x6HFBsz6fW', container=None, content=[TextBlock(citations=None, text="I'll help you find information about joining the course! Let me search for details about enrollment and joining policies.", type='text'), ToolUseBlock(id='toolu_01S54T5w3VwHgGJXccD4sp6Q', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment'}, name='search', type='tool_use'), ToolUseBlock(id='toolu_01VFMR9Sa6NranBt5au1SnRg', caller=DirectCaller(type='direct'), input={'query': 'how to enroll register course'}, name='search', type='tool_use'), ToolUseBlock(id='toolu_01VnBMqtjcZnHCMKvWiDjarW', caller=DirectCaller(type='direct'), input={'query': 'course admission requirements deadlines'}, name='search', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_inpu

In [ ]:
def make_call(call):
    args = call.input
    if call.name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown tool: {call.name}")
    result_json = json.dumps(result, indent=2)
    return {
        "type": "tool_result",
        "tool_use_id": call.id,
        "content": result_json,
    }

In [ ]:
messages.append({"role": "assistant", "content": response.content})

tool_results = []
for item in response.content:
    print(item)
    if item.type == "tool_use":
        print(f"FUNCTION CALL: {item.name}, {item.arguments}")
        tool_results.append(make_call(item))
    elif item.type == "text":
        print("ASSISTANT:")
        print(item.text)

if tool_results:
    messages.append({"role": "user", "content": tool_results})

TextBlock(citations=None, text="Great news! **Yes, you can join the course!** \n\nHere's what you need to know:\n\n✅ **You can join anytime** - The course materials (videos, notebooks, and GitHub resources) are available whenever you want to start.\n\n📋 **Certificate Requirements** - If you want to receive a certificate, you'll need to:\n- Submit your capstone project while the cohort is still accepting submissions\n- Complete required peer reviews\n\n**Important Note:** Certificates are only awarded for live cohorts. You can work through the material at your own pace, but project submission and peer reviews must happen while a live cohort is actively accepting them.\n\n**Getting Started:**\n1. Visit the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) to see deadlines and track your progress\n2. Check out the [L", type='text')
ASSISTANT:
Great news! **Yes, you can join the course!** 

Here's what you need to know:

✅ **You can join anytime**

In [ ]:
messages

[{'role': 'assistant',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you find information about joining the course! Let me search for details about enrollment and joining policies.", type='text'),
   ToolUseBlock(id='toolu_01S54T5w3VwHgGJXccD4sp6Q', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment'}, name='search', type='tool_use'),
   ToolUseBlock(id='toolu_01VFMR9Sa6NranBt5au

Putting all together - Agentic Tool Call Loop

In [ ]:
def agent_loop(
    instructions: str, question: str, model: str = "claude-haiku-4-5-20251001"
) -> str:

    messages = [
        {"role": "assistant", "content": instructions},
        {"role": "user", "content": question},
    ]

    iterations = 1

    while True:
        print(f"Starting iteration: {iterations}")
        has_function_calls = False

        response = anthropic_client.messages.create(
            model=model,
            max_tokens=200,
            messages=messages,
            tools=[search_tool],
        )

        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for item in response.content:
            print(item)
            if item.type == "tool_use":
                print(f"FUNCTION CALL: {item.name}, {item.input}")
                tool_results.append(make_call(item))
                has_function_calls = True

            elif item.type == "text":
                print("ASSISTANT:")
                last_answer = item.text

        if tool_results:
            messages.append({"role": "user", "content": tool_results})

        if not has_function_calls:
            break
        iterations += 1
        print(f"Iteration: {iterations}")
    return last_answer

Starting iteration: 1
TextBlock(citations=None, text="I'd like to help you with information about joining the course! Let me search for relevant details.", type='text')
ASSISTANT:
I'd like to help you with information about joining the course! Let me search for relevant details.
ToolUseBlock(id='toolu_01HipuB97e9gayfuMeomdcqs', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment'}, name='search', type='tool_use')
FUNCTION CALL: search, {'query': 'join course enrollment'}
ToolUseBlock(id='toolu_01WHcf8QHyuw8wc4VG5VtSZm', caller=DirectCaller(type='direct'), input={'query': 'how to enroll in course'}, name='search', type='tool_use')
FUNCTION CALL: search, {'query': 'how to enroll in course'}
ToolUseBlock(id='toolu_01DRbaqrA7TEzhnqfjM6WW4r', caller=DirectCaller(type='direct'), input={'query': 'course registration requirements'}, name='search', type='tool_use')
FUNCTION CALL: search, {'query': 'course registration requirements'}
Iteration: 2
Starting iteration: 2
Tex